# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

In [2]:
# =============================================================================
# CELL 1 — Install & vLLM load (run this FIRST every session)
# =============================================================================
!pip install vllm==0.8.5 sympy==1.13.1 antlr4-python3-runtime==4.11.1 -q

import os
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["VLLM_USE_V1"] = "0"

from vllm import LLM, SamplingParams
print("vLLM loaded!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.4/326.4 MB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 155.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.4/98.4 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 150.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.6/87.6 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.6/766.6 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 137.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 138.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 164.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/3

In [3]:
# =============================================================================
# CELL 2 — Mount Drive
# =============================================================================
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# =============================================================================
# CELL 3 — Imports & Config
# =============================================================================
import re
import csv
import json
import sys
from pathlib import Path
from typing import Optional
from transformers import AutoTokenizer
from tqdm.auto import tqdm

MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
DATA_PATH   = "/content/drive/MyDrive/competition_project/data/public.jsonl"
OUTPUT_PATH = "/content/drive/MyDrive/competition_project/results"

In [5]:
# =============================================================================
# CELL 4 — Load Dataset
# =============================================================================
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

Loaded 1126 questions  (375 MCQ, 751 free-form)


In [6]:
# =============================================================================
# CELL 5 — Prompts
# =============================================================================
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician competing in a math olympiad. "
    "Think carefully and systematically, checking your work at each step. "
    "Simplify all expressions fully to their simplest numerical or symbolic form. "
    "The question uses [ANS] as placeholders for answers. "
    "Put your final answer inside \\boxed{}. "
    "For multiple sub-answers, use a single \\boxed{} with comma separation "
    "in the same order as the [ANS] placeholders, e.g. \\boxed{3, 7}. "
    "Always double-check your arithmetic before giving the final answer."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician competing in a math olympiad. "
    "Solve the problem completely first, then check which answer choice matches. "
    "Output ONLY the letter of the single best answer inside \\boxed{}, e.g. \\boxed{C}. "
    "Do not write anything after the boxed letter."
)

def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"({lbl}) {opt.strip()}" for lbl, opt in zip(labels, options))
        user = (
            f"{question}\n\n"
            f"Answer Choices:\n{opts_text}\n\n"
            "Which option is correct? Output only \\boxed{{<letter>}}."
        )
        return SYSTEM_PROMPT_MCQ, user
    return SYSTEM_PROMPT_MATH, question


In [7]:
# =============================================================================
# CELL 6 — Load saved responses (skip inference)
# =============================================================================
responses = {}
backup_path = Path(OUTPUT_PATH) / "all_responses.jsonl"
with open(backup_path) as f:
    for line in f:
        rec = json.loads(line)
        responses[rec["id"]] = rec["response"]
print(f"Loaded {len(responses)} saved responses.")

Loaded 1126 saved responses.


In [8]:
# =============================================================================
# CELL 7 — Score
# =============================================================================
!cp /content/drive/MyDrive/competition_project/judger.py /content/judger.py
!cp /content/drive/MyDrive/competition_project/utils.py /content/utils.py
sys.path.insert(0, "/content")
from judger import Judger
judger = Judger(strict_extract=False)
print("Judger loaded!")

def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""

def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()

results = []
for item in tqdm(data, desc="Scoring"):
    response = responses.get(item.get("id"), "")
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

Judger loaded!


Scoring:   0%|          | 0/1126 [00:00<?, ?it/s]

Scoring complete. 1126 results.


In [9]:
# =============================================================================
# CELL 8 — Summary
# =============================================================================
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

EVALUATION RESULTS
  MCQ        :  182 /  375  (48.53%)
  Free-form  :  373 /  751  (49.67%)
  Overall    :  555 / 1126  (49.29%)


In [10]:
# =============================================================================
# CELL 9 — Save submission CSV
# =============================================================================
out_path = Path(OUTPUT_PATH) / "submission.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "response"])
    for item in data:
        resp = responses.get(item.get("id"), "")
        writer.writerow([item["id"], resp])

print(f"Saved {len(responses)} rows to {out_path}")

Saved 1126 rows to /content/drive/MyDrive/competition_project/results/submission.csv


In [11]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    dtype="bfloat16",
    gpu_memory_utilization=0.90,
    max_model_len=8192,
    trust_remote_code=True,
    max_num_seqs=32,
    enforce_eager=True,
    disable_async_output_proc=True,
)
print("Model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

INFO 06-01 00:07:44 [weight_utils.py:281] Time spent downloading weights for Qwen/Qwen3-4B-Thinking-2507: 18.648834 seconds


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


INFO 06-01 00:07:46 [loader.py:458] Loading weights took 2.49 seconds
INFO 06-01 00:07:47 [model_runner.py:1140] Model loading took 7.6065 GiB and 22.010859 seconds
INFO 06-01 00:07:49 [worker.py:287] Memory profiling takes 1.29 seconds
INFO 06-01 00:07:49 [worker.py:287] the current vLLM instance can use total_gpu_memory (79.25GiB) x gpu_memory_utilization (0.90) = 71.33GiB
INFO 06-01 00:07:49 [worker.py:287] model weights take 7.61GiB; non_torch_memory takes 0.09GiB; PyTorch activation peak memory takes 0.61GiB; the rest of the memory reserved for KV Cache is 63.02GiB.
INFO 06-01 00:07:49 [executor_base.py:112] # cuda blocks: 28680, # CPU blocks: 1820
INFO 06-01 00:07:49 [executor_base.py:117] Maximum concurrency for 8192 tokens per request: 56.02x
INFO 06-01 00:07:51 [llm_engine.py:437] init engine (profile, create kv cache, warmup model) took 4.58 seconds
Model loaded.


In [15]:
# private_data = [json.loads(line) for line in open("/content/drive/MyDrive/competition_project/data/private.jsonl")]
# print(f"Loaded {len(private_data)} private questions")

# prompts = []
# prompt_ids = []
# for item in private_data:
#     system, user = build_prompt(item["question"], item.get("options"))
#     prompt_text = tokenizer.apply_chat_template(
#         [{"role": "system", "content": system},
#          {"role": "user",   "content": user}],
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     prompts.append(prompt_text)
#     prompt_ids.append(item.get("id"))

# print(f"Generating responses for {len(prompts)} questions...")
# outputs = llm.generate(prompts, SamplingParams(max_tokens=8192, temperature=0.0))

# private_responses = {}
# for item_id, out in zip(prompt_ids, outputs):
#     private_responses[item_id] = out.outputs[0].text.strip()

# # Save backup
# backup_path = Path(OUTPUT_PATH) / "private_responses.jsonl"
# with open(backup_path, "w") as f:
#     for item_id, resp in private_responses.items():
#         f.write(json.dumps({"id": item_id, "response": resp}) + "\n")

# # Save submission CSV
# private_csv = Path(OUTPUT_PATH) / "private_submission.csv"
# with open(private_csv, "w", newline="") as f:
#     writer = csv.writer(f)
#     writer.writerow(["id", "response"])
#     for item in private_data:
#         resp = private_responses.get(item.get("id"), "")
#         writer.writerow([item["id"], resp])

# print(f"Saved private submission to {private_csv}")

In [16]:
!pip install peft trl bitsandbytes accelerate datasets -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 65.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.21.0 requires opentelemetry-api<=1.37.0,>=1.37.0, but you have opentelemetry-api 1.26.0 which is incompatible.
google-adk 1.21.0 requires opentelemetry-exporter-otlp-proto-http>=1.36.0, but you have opentelemetry-exporter-otlp-proto-http 1.26.0 which is incompatible.
google-adk 1.21.0 requires opentelemetry-sdk<=1.37.0,>=1.37.0, but you have opentelemetry-sdk 1.26.0 which is incompatible.


In [ ]:
!pip install pyarrow --upgrade -q
import os
os._exit(0)

In [ ]:
!pip install peft trl bitsandbytes accelerate datasets pyarrow --upgrade -q

# CELL B — Fine-tuning
import os
import json
import torch
from pathlib import Path
from datasets import load_dataset, concatenate_datasets
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

MODEL_ID   = "Qwen/Qwen3-4B-Thinking-2507"
OUTPUT_DIR = "/content/drive/MyDrive/competition_project/qlora_model"  # save to Drive!
MAX_SEQ_LEN = 2048
NUM_EPOCHS = 2
BATCH_SIZE = 4
GRAD_ACCUM = 4
LEARNING_RATE = 2e-4
MAX_TRAIN_SAMPLES = 20000

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def format_metamath(example):
    messages = [
        {"role": "system", "content": "You are an expert mathematician. Solve the problem step-by-step. Put your final answer inside \\boxed{}."},
        {"role": "user", "content": example.get("query", "")},
        {"role": "assistant", "content": example.get("response", "")},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

def format_math_hendrycks(example):
    messages = [
        {"role": "system", "content": "You are an expert mathematician. Solve the problem step-by-step. Put your final answer inside \\boxed{}."},
        {"role": "user", "content": example.get("problem", "")},
        {"role": "assistant", "content": example.get("solution", "")},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

def format_gsm8k(example):
    answer_text = example.get("answer", "")
    if "####" in answer_text:
        parts = answer_text.split("####")
        answer_text = f"{parts[0].strip()}\n\nThe answer is \\boxed{{{parts[1].strip()}}}"
    messages = [
        {"role": "system", "content": "You are an expert mathematician. Solve the problem step-by-step. Put your final answer inside \\boxed{}."},
        {"role": "user", "content": example.get("question", "")},
        {"role": "assistant", "content": answer_text},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

all_datasets = []

print("Loading MetaMathQA...")
try:
    metamath = load_dataset("meta-math/MetaMathQA", split="train")
    metamath = metamath.shuffle(seed=42).select(range(min(12000, len(metamath))))
    metamath = metamath.map(format_metamath, remove_columns=metamath.column_names)
    all_datasets.append(metamath)
    print(f"  MetaMathQA: {len(metamath)} examples")
except Exception as e:
    print(f"  MetaMathQA failed: {e}")

print("Loading MATH (Hendrycks)...")
try:
    math_ds = load_dataset("hendrycks/competition_math", split="train", trust_remote_code=True)
    math_ds = math_ds.map(format_math_hendrycks, remove_columns=math_ds.column_names)
    all_datasets.append(math_ds)
    print(f"  MATH: {len(math_ds)} examples")
except Exception as e:
    print(f"  MATH failed: {e}")

print("Loading GSM8K...")
try:
    gsm = load_dataset("openai/gsm8k", "main", split="train")
    gsm = gsm.map(format_gsm8k, remove_columns=gsm.column_names)
    all_datasets.append(gsm)
    print(f"  GSM8K: {len(gsm)} examples")
except Exception as e:
    print(f"  GSM8K failed: {e}")

train_dataset = concatenate_datasets(all_datasets).shuffle(seed=42)
if len(train_dataset) > MAX_TRAIN_SAMPLES:
    train_dataset = train_dataset.select(range(MAX_TRAIN_SAMPLES))
print(f"\nTotal training examples: {len(train_dataset)}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=64,
    lora_alpha=128,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    logging_steps=25,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    bf16=True,
    max_length=MAX_SEQ_LEN,          # was max_seq_length
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataset_text_field="text",
    packing=True,
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\nTraining complete! Model saved to {OUTPUT_DIR}")

Loading MetaMathQA...


Map:   0%|          | 0/12000 [00:00<?, ? examples/s]

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'hendrycks/competition_math' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'hendrycks/competition_math' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


  MetaMathQA: 12000 examples
Loading MATH (Hendrycks)...
  MATH failed: Dataset 'hendrycks/competition_math' doesn't exist on the Hub or cannot be accessed.
Loading GSM8K...


Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

...

Adding EOS to train dataset:   0%|          | 0/19473 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/19473 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/19473 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Step,Training Loss
